# Qwen3.8 × ARC Phase A — Quick Colab (single model load)

**目的:** 3問のdevelopmentパイロット → 既存の**固定200問**を同じGPUモデルで連続実行します。モデルを問題ごとに再ロードしたり、Macでmanifestを事前登録したりする必要はありません。

1. Colabで **H100 80GB（推奨）**、または **A100 80GB** を選択。
2. `MyDrive/qwen-colab/inputs/arc-agi_training_challenges.json` を用意（**solutionsはアップロードしない**）。
3. 上からセルを順に実行。パイロットを確認後、Phase Aのセルを実行。
4. 結果は `MyDrive/qwen-colab/runs/` に**1問ずつ保存**。Colabが切れたらセルを再実行して続きから再開。

**モデル:** H100では `Qwen3.8-27B-FP8`、A100 80GBでは `Qwen3.8-27B` (BF16) を自動選択。これらは同じ実験として混在させず、実行フォルダを分けます。A100 40GBは対象外です。

**研究上の扱い:** これは元の `lt_single` のmanifest/プロセス分離を置き換える **quick exploratory protocol** です。元の厳密なPhase Aランと混ぜて比較しないでください。3問は開発用、200問は元リポジトリの割当を固定使用します。正解照合はMacなどのローカルで別途行います。Colab接続中はcompute unitsが消費されます。

In [ ]:
# 1. 設定（最初はここだけ確認）
REPO_URL = 'https://github.com/morshoto/qwen-meetup-2.git'
REPO_REVISION = 'cbba28c04dfa28fb900b92087e6fd5214da7e6b3'  # 固定200問リストの参照元
DRIVE_ROOT = '/content/drive/MyDrive/qwen-colab'
RUN_LABEL = 'qwen38-quick-phasea-v1'  # 設定を変えた新実験は別名にする
VLLM_VERSION = '0.33.0'

MAX_MODEL_LEN = 18432
MAX_NEW_TOKENS = 4096
TEMPERATURE = 0.6
SEED_BASE = 20261111
GPU_MEMORY_UTILIZATION = 0.90

PILOT_LIMIT = 3
PHASE_A_LIMIT_PER_LAUNCH = 200       # 1回の実行で処理する最大の未完了問題数
PHASE_A_MAX_MINUTES = 240           # 次の問題を開始する前にチェック（実際のColab課金上限ではない）
REQUIRE_VALID_PILOT = True          # 3問それぞれに少なくとも1つvalidな予測が必要
FORMAT_RECOVERY = True              # strict JSONが失敗したときだけ、安全な形式修復を試す

## 2. Colab環境を準備

初回のPythonパッケージインストールは必要です。**以前のnotebookのCUDAツールキット手作業や31GBの全ファイル再ハッシュは省略**します。`vllm` のインストールでColabのPython環境が大きく変わった場合は、ランタイムを再起動し、このセル以降を再実行してください。

In [ ]:
import subprocess, sys, importlib.metadata
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                f'vllm=={VLLM_VERSION}', 'transformers>=5.8.0', 'huggingface_hub'], check=True)
print('vLLM:', importlib.metadata.version('vllm'))

## 3. Driveと固定のARCタスクを読み込む

元のPhase A **200問のIDと順序**・development **3問のID**・promptをGitHubの指定コミットから読みます。入力データのSHA-256を元の割当と照合し、**テスト問題の正解出力はプロンプトから除外**します。`arc-agi_training_solutions.json` はColabで読みません。

In [ ]:
import os, sys, json, hashlib, subprocess, re, types, time
from pathlib import Path
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

repo = Path('/content/qwen-meetup-2')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '-q', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', REPO_REVISION], check=True)
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip() == REPO_REVISION
sys.path.insert(0, str(repo))
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
src_pkg = types.ModuleType('src')
src_pkg.__path__ = [str(repo / 'src')]
sys.modules['src'] = src_pkg

from src.arc.lt_live_backend import PROMPT
from src.arc.lt_direct_format import normalize_direct
from src.arc.data import validate_grid

challenges_file = Path(DRIVE_ROOT) / 'inputs' / 'arc-agi_training_challenges.json'
if not challenges_file.is_file():
    raise FileNotFoundError(f'{challenges_file} がありません。solutionsではなくtraining_challengesをDriveへ置いてください。')
challenge_bytes = challenges_file.read_bytes()
challenge_sha = hashlib.sha256(challenge_bytes).hexdigest()
proposal = json.loads((repo / 'configs/lt.phase-a.comparison-proposal.json').read_text())
development = json.loads((repo / 'configs/lt.phase-a.development.json').read_text())
assert challenge_sha == proposal['dataset_sha256'], '入力データのSHAが固定Phase Aのデータと一致しません'
phase_ids = proposal['task_ids']
dev_ids = development['task_ids']
assert len(phase_ids) == 200 and len(set(phase_ids)) == 200
assert len(dev_ids) == 3 and not (set(dev_ids) & set(phase_ids))
all_tasks = json.loads(challenge_bytes)

def clean_task(task_id):
    source = all_tasks[task_id]
    # testのoutputがファイルに存在しても絶対にモデルへ送らない
    train = [{'input':validate_grid(x['input']), 'output':validate_grid(x['output'])} for x in source['train']]
    tests = [{'input':validate_grid(x['input'])} for x in source['test']]
    assert train and tests
    return {'train':train,'test':tests}

tasks = {task_id:clean_task(task_id) for task_id in dev_ids + phase_ids}
print('検証OK | challenge SHA:',challenge_sha[:16], '| dev:',len(dev_ids),'| Phase A:',len(phase_ids))
print('プロンプトSHA:', hashlib.sha256(PROMPT.encode()).hexdigest()[:16])

## 4. GPUに合ったモデルを選び、実験を登録

まず**GPUの互換性、トークン長、再開時の設定一致**を確認します。H100/BlackwellならFP8、A100 80GBならBF16を使用。新しい設定を使う場合は `RUN_LABEL` を変更してください。

In [ ]:
import torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

gpu = torch.cuda.get_device_properties(0)
gpu_name = gpu.name
gpu_gib = gpu.total_memory / 1024**3
print(f'GPU: {gpu_name} | {gpu_gib:.1f} GiB | compute capability {gpu.major}.{gpu.minor}')
if gpu.major >= 9:
    MODEL_ID = 'Qwen/Qwen3.8-27B-FP8'
    MODEL_REVISION = 'c3a462cc3d38cf351f3b26cbd763469eeb74ef81'
    precision = 'fp8'
elif 'A100' in gpu_name and gpu_gib >= 70:
    MODEL_ID = 'Qwen/Qwen3.8-27B'
    MODEL_REVISION = '37561816e66b5b9153b1b64c10968c18c1e6f4a0'
    precision = 'bf16'
else:
    raise RuntimeError('このノートブックはH100級（FP8）またはA100 80GB（BF16）が必要です。')

# tokenizerだけを先に取得。モデル本体を落とす前にcontext超過を確認。
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False)
def prompt_for(task_id):
    payload = {'instruction': PROMPT, 'task': tasks[task_id]}
    content = json.dumps(payload, sort_keys=True, separators=(',',':'))
    return tokenizer.apply_chat_template([{'role':'user','content':content}],
             tokenize=False, add_generation_prompt=True, enable_thinking=False)
prompts = {task_id: prompt_for(task_id) for task_id in dev_ids + phase_ids}
prompt_lengths = {task_id:len(tokenizer.encode(p,add_special_tokens=False)) for task_id,p in prompts.items()}
longest_task = max(prompt_lengths,key=prompt_lengths.get)
assert prompt_lengths[longest_task] + MAX_NEW_TOKENS <= MAX_MODEL_LEN, (
    f'context超過: {longest_task}: {prompt_lengths[longest_task]}+{MAX_NEW_TOKENS}>{MAX_MODEL_LEN}')

model_key = 'fp8-hopper' if precision == 'fp8' else 'bf16-a100'
assert re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9_-]*',RUN_LABEL)
run_dir = Path(DRIVE_ROOT) / 'runs' / f'{RUN_LABEL}-{model_key}'
run_dir.mkdir(parents=True, exist_ok=True)

def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(',',':'), ensure_ascii=False, allow_nan=False).encode()
def sha_obj(value):
    return hashlib.sha256(canonical(value)).hexdigest()
def atomic_json(path, value):
    # One-task checkpoints. The temporary file belongs to the same Drive folder.
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_bytes(canonical(value) + b'\n')
    os.replace(tmp, path)

manifest = {
    'protocol':'qwen38-quick-exploratory-v1', 'repo_revision':REPO_REVISION,
    'challenge_sha256':challenge_sha, 'allocation_sha256':sha_obj(proposal),
    'prompt_sha256':hashlib.sha256(PROMPT.encode()).hexdigest(),
    'dev_task_ids':dev_ids,'phase_a_task_ids':phase_ids,
    'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'precision':precision,
    'vllm_version':VLLM_VERSION,
    'max_model_len':MAX_MODEL_LEN,'max_new_tokens':MAX_NEW_TOKENS,
    'temperature':TEMPERATURE,'seed_base':SEED_BASE,
    'gpu_memory_utilization':GPU_MEMORY_UTILIZATION,
    'format_recovery':FORMAT_RECOVERY,'candidate_count':2,'thinking':'off',
}
manifest['sha256']=sha_obj(manifest)
manifest_path=run_dir/'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, (
        '既存の実験と設定が異なります。RUN_LABELを変更してください。')
else:
    atomic_json(manifest_path, manifest)

print('モデル:',MODEL_ID,'| prompt最大:',prompt_lengths[longest_task], 'tokens')
print('保存先:',run_dir)
print('modelを1回ロードした後はランタイムが切れるまで保持します。')

## 5. モデルを**1回だけ**ロード

ここで大きなモデルのダウンロードと初回GPUロードを行います（H100 FP8: 約31GB、A100 BF16: 約56GB）。同じColabセッションでは再度実行する必要はありません。再開時は同じDrive上の結果を読み、**未実行タスクだけ**処理します。初回ロードと互換性は現時点では実GPU未検証です。

In [ ]:
from vllm import LLM, SamplingParams
assert importlib.metadata.version('vllm') == VLLM_VERSION
model_path = snapshot_download(MODEL_ID, revision=MODEL_REVISION)
llm = LLM(
    model=model_path, trust_remote_code=False,
    dtype='bfloat16', max_model_len=MAX_MODEL_LEN,
    tensor_parallel_size=1, language_model_only=True,
    enforce_eager=True, max_num_seqs=1,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
)
print('モデルロード完了。ここから先は各問題で再ロードしません。')

## 6. 推論・厳密な構造チェック・自動再開

出力は `normalize_direct()` で厳密に検査。失敗時のみ、`FORMAT_RECOVERY=True` なら**コードフェンス除去**・**1問の単一グリッドをリストへ包む**という構造上の修正を行います。修復の有無はすべて記録します。**推論の正誤判定はしません。**

**注意:** モデルを再ロードしない代わりに、元実装の「タスク単位の別プロセスでの強制タイムアウト」は廃止しています。時間予算は**各タスク開始前**に確認し、実行中の問題は途中終了しません。GPU OOMなどでセルが失敗しても保存済みタスクは残ります。

In [ ]:
import json, time, traceback

def parse_prediction(raw, count):
    try:
        return normalize_direct(raw, count), 'strict', None
    except (ValueError, TypeError) as original:
        if not FORMAT_RECOVERY:
            return None, 'invalid', str(original)
    value = raw.strip()
    repairs=[]
    if value.startswith('<think>') and value.count('</think>') == 1:
        value = value.split('</think>',1)[1].strip()
        repairs.append('remove_thinking_block')
    if value.startswith('```') and value.endswith('```'):
        lines = value.splitlines()
        if len(lines)>=3 and lines[0].strip().lower() in ('```','```json') and lines[-1].strip()=='```':
            value='\n'.join(lines[1:-1]).strip()
            repairs.append('remove_code_fence')
    try:
        return normalize_direct(value, count), '+'.join(repairs) or 'strict', None
    except (ValueError, TypeError) as cleaned_error:
        if count == 1:
            try:
                parsed=json.loads(value)
                if isinstance(parsed,dict) and set(parsed)=={'output'}:
                    parsed=parsed['output']
                    repairs.append('unwrap_output_key')
                grid=validate_grid(parsed)
                return [grid], '+'.join(repairs+['wrap_single_grid']), None
            except (ValueError, TypeError, KeyError):
                pass
        return None, 'invalid', str(cleaned_error)


def checkpoint_path(kind, task_id):
    return run_dir / kind / f'{task_id}.json'

def load_checkpoint(kind, task_id):
    path=checkpoint_path(kind,task_id)
    if not path.exists():
        return None
    result=json.loads(path.read_text())
    if result.get('task_id')!=task_id or result.get('manifest_sha256')!=manifest['sha256']:
        raise ValueError('checkpoint identity mismatch: '+str(path))
    if len(result.get('candidates',[]))!=2:
        raise ValueError('checkpoint damaged: '+str(path))
    return result

def infer_task(task_id):
    started=time.monotonic()
    count=len(tasks[task_id]['test'])
    candidates=[]
    for i in (0,1):
        sampling=SamplingParams(temperature=TEMPERATURE,top_p=1.0,top_k=-1,
                                seed=SEED_BASE+i,max_tokens=MAX_NEW_TOKENS)
        answer=llm.generate([prompts[task_id]],sampling,use_tqdm=False)[0]
        completion=answer.outputs[0]
        raw=completion.text
        parsed,mode,error=parse_prediction(raw,count)
        candidates.append({
            'candidate_index':i,'outputs':parsed,'valid':parsed is not None,
            'parse_mode':mode,'format_error':error,'raw':raw,
            'prompt_tokens':len(answer.prompt_token_ids),
            'completion_tokens':len(completion.token_ids),
            'finish_reason':completion.finish_reason,
        })
    return {
        'task_id':task_id,'manifest_sha256':manifest['sha256'], 'status':'complete',
        'test_count':count,'gpu_name':gpu_name,'runtime_seconds':round(time.monotonic()-started,3),
        'candidates':candidates,
    }

def run_tasks(kind, ids, *, limit, minutes):
    started=time.monotonic()
    attempted=0
    for task_id in ids:
        if load_checkpoint(kind,task_id) is not None:
            continue
        if attempted>=limit or time.monotonic()-started>=minutes*60:
            print('実行数または実行時間の上限に達しました。')
            break
        print(f'[{kind}] {attempted+1}/{limit} | {task_id} ...',flush=True)
        try:
            result=infer_task(task_id)
        except Exception as exc:
            print('GPU/推論エラー:',type(exc).__name__,str(exc)[:800])
            print('このタスクは未完了です。保存済みのタスクは後で再開できます。')
            raise
        atomic_json(checkpoint_path(kind,task_id),result)
        attempted+=1
        print('  valid candidates:',sum(c['valid'] for c in result['candidates']),'/2',
              '| seconds:',round(result['runtime_seconds'],1),flush=True)
    done=[load_checkpoint(kind,t) for t in ids]
    done=[r for r in done if r is not None]
    valid_one=sum(any(c['valid'] for c in r['candidates']) for r in done)
    print(f'{kind}: {len(done)}/{len(ids)} tasks saved | at least one valid: {valid_one}/{len(done)}')
    return done

## 7. Developmentパイロット（3問）

既存の開発3問を使い、**正答率ではなく構造上のvalidityだけ**を確認します。3問の結果はPhase Aには含まれません。ここでモデルが例外終了・3問すべてinvalidなら、200問を始める前に原因を調べてください。

In [ ]:
pilot_records = run_tasks('pilot', dev_ids, limit=PILOT_LIMIT, minutes=60)
for r in pilot_records:
    print(r['task_id'], [(c['valid'],c['parse_mode'],(c['format_error'] or '')[:120])
                         for c in r['candidates']])

## 8. Phase A — 固定200問を実行

**ここが本番実行セルです。** Phase A開始前にpilotの有効形式を確認し、200問を同一vLLMインスタンスで処理します。1問が終わるごとにDriveへ保存します。停止後はロードセルまで再実行し、同じセルで続きから再開可能。既存のinvalidな出力を黙って再試行しません（別ランでのみ再実験）。

In [ ]:
pilot_records=[load_checkpoint('pilot',t) for t in dev_ids]
if REQUIRE_VALID_PILOT:
    assert all(r is not None and any(c['valid'] for c in r['candidates']) for r in pilot_records), (
        'pilot未完了かinvalidです。200問実行前にrawとformat_errorを確認してください。')
phase_records=run_tasks('phase_a',phase_ids,
    limit=PHASE_A_LIMIT_PER_LAUNCH, minutes=PHASE_A_MAX_MINUTES)

## 9. 実行状況と採点用JSONを作成（いつでも再実行可）

Driveの各タスクの結果を1ファイルへまとめます。**validはJSON形式が正しいという意味であり、ARC正解率ではありません。** 最終採点はsolutionsを保持するローカル環境で、同梱の `score_qwen38_quick_phasea.py` を使用します。

In [ ]:
phase_records=[load_checkpoint('phase_a',t) for t in phase_ids]
saved=[r for r in phase_records if r is not None]
report={
    'protocol':manifest['protocol'],'manifest':manifest,
    'complete':len(saved)==len(phase_ids),
    'total_tasks':len(phase_ids),'attempted_tasks':len(saved),
    'valid_first':sum(r['candidates'][0]['valid'] for r in saved),
    'valid_either':sum(any(c['valid'] for c in r['candidates']) for r in saved),
    'task_results':{r['task_id']:r for r in saved},
}
output_path=run_dir/'phase_a_results.json'
atomic_json(output_path, report)
print(f"saved={len(saved)}/{len(phase_ids)} | first valid={report['valid_first']} | either valid={report['valid_either']}")
print('結果:',output_path)
print('Phase Aの正答率ではありません。ローカルの正解データで採点してください。')

## ローカル採点（Colabでは実行しない）

1. Driveの `phase_a_results.json` をMacにダウンロード。
2. このノートブックと一緒に配布した `score_qwen38_quick_phasea.py` をMacへ置く。
3. **Macにだけ保存した** `arc-agi_training_solutions.json` で採点:

```bash
python score_qwen38_quick_phasea.py \
  --results phase_a_results.json \
  --solutions /path/to/arc-agi_training_solutions.json \
  --output qwen38_quick_phasea_score.json
```

採点結果には **pass@1 / pass@2、200問に対する分母、未完了問題数** を表示します。これはARC-AGI-2公式120問の評価ではなく、学習タスクから固定した200問の **exploratory Phase A** です。モデルの重みは一切更新していません（RSIのPhase Bは別工程）。

**後片付け:** 終了したらColabのランタイムを切断して、GPUの利用を止めてください。